# Uudelleenkäytettävän vakuutusmatemaattisen hinnoittelukirjaston rakentaminen PROC FCMP:llä

## Tiivistelmä

Vahinkovakuutusyhtiö kapseloi ydinhinnoittelunsa laskennan — puhdas vakuutusmaksu, kulu-/riskikuormitus, rajoitetun vaihtelun uskottavuuspainotus ja diskontattu vastuuvelan arviointi — omiksi funktioikseen ja monituloisteiseksi aliohjelmaksi **PROC FCMP** -menettelyssä. Käännetty kirjasto rekisteröidään `CMPLIB=`-järjestelmäasetuksella ja sitä kutsutaan sitten rivi riviltä DATA-askeleesta, joka hinnoittelee synteettisen 100 vakuutuksen portfolion. Jokainen tässä muistikirjassa hinnoiteltu luku — uskottavuuskerroin `z`, painotettu puhdas vakuutusmaksu, veloitettu vakuutusmaksu ja nykyarvoon diskontattu korvausvastuu — tuotetaan näillä käännetyillä rutiineilla, ei rivin sisäisellä laskennalla. Tulos: toteutunut vahinkosuhde asettuu tasolle 60.5 % (maaseutu), 55.8 % (esikaupunki) ja 63.6 % (kaupunki) — mukavasti alle 100 % joka segmentissä, mikä vahvistaa, että kuormitettu vakuutusmaksu kattaa odotetun vahingon rating-askeleen pysyessä siistinä ja tarkastettavana.

## Tietolähteet

| Aineisto | Rivit | Kuvaus | Avainmuuttujat |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Synteettinen voimassa oleva vahinkovakuutusportfolio, generoitu `rand()`-funktiolla | `policy_id`, `region` (kaupunki/esikaupunki/maaseutu), `years_insured`, `exposure` (autovuosia), `claim_count` (Poisson), `incurred_loss` (gamma-vahinkosuuruus x lukumäärä), `class_pure_prem` (manuaalinen luokkamaksu alueittain) |

DATA-askel silmukoi laajemman `policy_id`-alueen yli, mutta tämä ympäristö toimii lisenssittömässä tilassa, joten tuloste on rajattu ensimmäisiin **100 havaintoon** — alla oleva hinnoiteltu kirja kuvaa näitä 100 vakuutusta.

# Uudelleenkäytettävän vakuutusmatemaattisen hinnoittelukirjaston rakentaminen PROC FCMP:llä

Aktuaarit toistavat samoja laskelmia hinnoittelu-, vastuuvelka- ja raportointitöissä: muuntavat vahingot *puhtaaksi vakuutusmaksuksi*, soveltavat *kulu- ja riskikuormituksia* veloitetun vakuutusmaksun saavuttamiseksi, painottavat yksittäisen vakuutuksen omaa kokemusta luokkamaksulla *uskottavuuden* avulla ja *diskonttaavat* tulevat kassavirrat nykyarvoon. Näiden kaavojen uudelleenkirjoittaminen jokaisessa DATA-askeleessa altistaa copy-paste-virheille ja tekee oletusten muuttamisesta työlästä.

**PROC FCMP** (SAS:n funktiokääntäjä) antaa meidän määritellä jokaisen kaavan kerran nimettynä funktiona tai aliohjelmana, tallentaa käännetyt rutiinit kirjastoon ja kutsua niitä sitten kuin mitä tahansa SAS:n sisäänrakennettua funktiota. Tässä muistikirjassa:

1. Käännämme pienen vakuutusmatemaattisen funktiokirjaston `PROC FCMP`:llä.
2. Rekisteröimme sen istunnolle `CMPLIB=`-järjestelmäasetuksella.
3. Generoimme synteettisen vahinkovakuutusportfolion.
4. Hinnoittelemme jokaisen vakuutuksen kutsumalla omia funktioitamme ja monituloisteista aliohjelmaa yhdestä DATA-askeleesta.
5. Tiivistämme ja tulkitsemme hinnoitellun portfolion.

## Vaihe 1 — Synteettisen vakuutusportfolion generointi

Simuloimme voimassa olevien autovakuutusten kirjan (ensimmäiset 100 hinnoitellaan alla lisenssittömän tilan rajoituksen puitteissa). Jokainen vakuutus kuuluu hinnoittelualueeseen, jolla on oma manuaalinen *puhdas vakuutusmaksu* (odotettu vahinko autovuotta kohden). Vahinkolukumäärät noudattavat exposurella skaalattua Poisson-prosessia, ja vahinkosuuruudet noudattavat gamma-jakaumaa, joten `incurred_loss` on realistinen yhdistetty (Poisson x gamma) vahinko. `years_insured` ohjaa myöhemmin uskottavuuspainoa. Kiinteä siemenluku `call streaminit`-funktion kautta tekee datasta toistettavan.

In [1]:
TIEDOT portfolio;
    CALL streaminit(20260531);
    PITUUS region $16;
    TEE policy_id = 10001 ASTI 12000;
        /* Hinnoittelualueen määrittäminen */
        u = rand('uniform');
        JOS u < 0.40 NIIN TEE; region = 'kaupunki';    base_pp = 820; lambda = 0.18; LOPPU;
        MUUTEN JOS u < 0.75 NIIN TEE; region = 'esikaupunki'; base_pp = 540; lambda = 0.11; LOPPU;
        MUUTEN TEE; region = 'maaseutu';    base_pp = 360; lambda = 0.07; LOPPU;

        /* Vakuutusaika (vuosia) ja exposure (autovuosia) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Yhdistetty vahinkoprosessi: Poisson-frekvenssi, gamma-vahinkosuuruus */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        TEE c = 1 ASTI claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        LOPPU;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manuaalinen luokan puhdas vakuutusmaksu tälle exposurelle */
        class_pure_prem = round(base_pp * exposure, 0.01);

        TULOSTE;
    LOPPU;
    SÄILYTÄ policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=portfolio(obs=8) noobs;
    NIMIKE policy_id='Vakuutuksen tunnus' region='Alue' years_insured='Vakuutusaika (v)'
          exposure='Exposure (autovuosia)' claim_count='Vahinkojen lukumäärä'
          incurred_loss='Toteutunut vahinko' class_pure_prem='Luokan puhdas maksu';
    OTSIKKO 'Ensimmäiset 8 simuloitua vakuutusta';
SUORITA;

                                          Ensimmäiset 8 simuloitua vakuutusta                                           

Vakuutuksen tunnus         Alue  Vakuutusaika (v)  Exposure (autovuosia)     Vahinkojen lukumäärä  Toteutunut vahinko  Luokan puhdas maksu
             10001  maaseutu                    5                   1.36                        0                   0                489.6
             10002  kaupunki                    8                   0.97                        1             3432.28                795.4
             10003  kaupunki                    2                   1.53                        2             7155.92               1254.6
             10004  esikaupunki                 9                    2.4                        0                   0                 1296
             10005  maaseutu                    5                   0.99                        0                   0                356.4
             10006  esikaupunki             


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.42 seconds
  cpu   0.42 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Vaihe 2 — Vakuutusmatemaattisen funktiokirjaston kääntäminen

Nyt muistikirjan ydin. `PROC FCMP` asetuksella `OUTLIB=work.actfuncs.pricing` kääntää neljä funktiota ja yhden aliohjelman `work.actfuncs`-aineiston `pricing`-pakettiin:

- **`pure_premium`** — havaittu vahinko exposure-yksikköä kohden (frekvenssi x vahinkosuuruus yhdistettynä).
- **`credibility_z`** — rajoitetun vaihtelun uskottavuuskerroin `Z = sqrt(n / (n + k))`, jossa `n` on vakuutuksen exposure-vuodet ja `k` on säätövakio.
- **`charged_premium`** — soveltaa suhteellista riskikuormitusta ja kiinteää kuluastetta vahinkokustannukseen veloitetun vakuutusmaksun saavuttamiseksi.
- **`pv_reserve`** — tulevan korvausmaksun nykyarvo, `FV / (1+r)**t`, jota käytetään korvausvastuiden diskonttaamiseen.
- **`blend_premium`** (ALIOHJELMA) — käyttää `OUTARGS`-määrettä palauttaakseen *kaksi* arvoa kerralla: uskottavuuspainotetun puhtaan vakuutusmaksun ja käytetyn uskottavuuskertoimen, jotta DATA-askel saa molemmat yhdellä kutsulla.

Jokainen rutiini päätetään `ENDSUB`-lauseella, ja aliohjelma nimeää kirjoitettavat argumenttinsa `OUTARGS`-määreellä.

In [2]:
PROSEDUURI fcmp outlib=work.actfuncs.pricing;

    /* Havaittu puhdas vakuutusmaksu: vahinkokustannus exposure-yksikköä kohden */
    function pure_premium(loss, exposure);
        JOS exposure <= 0 NIIN RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Rajoitetun vaihtelun uskottavuus: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        JOS n_years <= 0 NIIN RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Veloitettu vakuutusmaksu = vahinkokustannus korotettuna riskikuormituksella + kuluilla */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        JOS expense_ratio >= 1 NIIN RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Tulevan korvausmaksun nykyarvo diskontattuna t vuotta korolla r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Uskottavuuspainotus: palauttaa painotetun puhtaan vakuutusmaksun JA käytetyn Z:n */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

SUORITA;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Vaihe 3 — Kirjaston rekisteröinti CMPLIB=-asetuksella

Rutiinien kääntäminen ei riitä; SAS:lle on kerrottava, mistä ne löytyvät, kun myöhempi DATA-askel tai proseduuri viittaa nimeen, jota se ei tunnista sisäänrakennetuksi. `CMPLIB=`-järjestelmäasetus osoittaa käännetyn koodin sisältävään aineistoon (ei kolmitasoiseen pakettinimeen). Tämän `OPTIONS`-lauseen jälkeen jokainen yllä oleva funktio ja aliohjelma on kutsuttavissa nimellä.

In [3]:
ASETUKSET cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Vaihe 4 — Portfolion hinnoittelu mukautetuilla funktioilla

Hinnoitteleva DATA-askel on nyt lähes vapaa aritmetiikasta — aktuaarinen tarkoitus näkyy suoraan funktioiden nimistä. Jokaiselle vakuutukselle:

1. Lasketaan vakuutuksen oma havaittu puhdas vakuutusmaksu `pure_premium`-funktiolla.
2. Kutsutaan `blend_premium`-aliohjelmaa uskottavuuspainottamaan sitä alueellista luokkamaksua vasten, palauttaen sekä painotetun vahinkokustannuksen että käytetyn uskottavuuskertoimen `z` `OUTARGS`-määreen kautta.
3. Korotetaan painotettu vahinkokustannus veloitetuksi vakuutusmaksuksi 12 %:n riskikuormituksella ja 25 %:n kuluasteella `charged_premium`-funktiolla.
4. Arvioidaan vielä avoin korvausvastuu 35 %:na vakuutuksen toteutuneesta vahingosta ja diskontataan se kolme vuotta 4 %:n korolla nykyarvoon `pv_reserve`-funktiolla.

Huomaa, että aliohjelman tulostusargumentit (`blended_pp`, `z`) on alustettava ennen `CALL`-kutsua. Vastuuvelan nykyarvo vaihtelee vakuutuksittain, koska se määräytyy kunkin vakuutuksen todellisen toteutuneen vahingon mukaan — vahingottomilla vakuutuksilla on nollavastuu, joten myös niiden `reserve_pv` on nolla.

In [4]:
TIEDOT rated;
    ASETA portfolio;

    /* 1. Vakuutuksen oma vahinkokokemus puhtaana vakuutusmaksuna */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Oman kokemuksen uskottavuuspainotus luokkamaksua vasten.
          k = 4 exposure-vuotta lähes täydelle uskottavuudelle. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Painotetun vahinkokustannuksen (autovuotta kohden) muuntaminen veloitetuksi vakuutusmaksuksi */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Vielä avoin korvausvastuu = 35 % toteutuneesta vahingosta, oletetaan
          selvitettävän 3 vuodessa; diskontataan nykyarvoon 4 %:n korolla. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    SÄILYTÄ policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=rated(obs=10) noobs;
    NIMIKE policy_id='Vakuutuksen tunnus' region='Alue' years_insured='Vakuutusaika (v)'
          exposure='Exposure (autovuosia)' own_pp='Oma puhdas maksu' blended_pp='Painotettu puhdas maksu'
          z='Uskottavuuskerroin (z)' premium='Veloitettu vakuutusmaksu' reserve_pv='Vastuuvelka (nykyarvo)';
    OTSIKKO 'Hinnoiteltu portfolio (ensimmäiset 10 vakuutusta)';
    MUUTTUJA policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
SUORITA;

                                   Hinnoiteltu portfolio (ensimmäiset 10 vakuutusta)                                    

Vakuutuksen tunnus         Alue  Vakuutusaika (v)  Exposure (autovuosia)  Oma puhdas maksu  Painotettu puhdas maksu  Uskottavuuskerroin (z)  Veloitettu vakuutusmaksu  Vastuuvelka (nykyarvo)
             10001  maaseutu                    5                   1.36                 0                    91.67                   0.745                    186.18                       0
             10002  kaupunki                    8                   0.97           3538.43                  3039.59                   0.816                   4402.95                 1067.95
             10003  kaupunki                    2                   1.53           4677.07                  3046.88                   0.577                   6961.51                 2226.55
             10004  esikaupunki                 9                    2.4                 0                    90.69   


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Vaihe 5 — Hinnoitellun kirjan tiivistäminen

Kun jokainen vakuutus on hinnoiteltu saman käännetyn kirjaston kautta, voimme koota portfolion yhteen alueittain. Raportoimme keskimääräisen veloitetun vakuutusmaksun, keskimääräisen uskottavuuskertoimen, kokonaistoteutuneen vahingon ja nykyarvoon diskontatun kokonaiskorvausvastuun, ja johdamme sitten toteutuneen vahinkosuhteen segmenteittäin, jotta näemme, kattaako kuormitettu vakuutusmaksu odotetun vahingon.

In [5]:
PROSEDUURI KESKIARVOT TIEDOT=rated mean sum maxdec=2 nonobs;
    LUOKKA region;
    MUUTTUJA premium z incurred_loss reserve_pv;
    NIMIKE region='Alue' premium='Veloitettu vakuutusmaksu' z='Uskottavuuskerroin (z)'
          incurred_loss='Toteutunut vahinko' reserve_pv='Vastuuvelka (nykyarvo)';
    OTSIKKO 'Portfolion yhteenveto hinnoittelualueittain';
SUORITA;

/* Toteutunut vahinkosuhde = toteutunut vahinko / veloitettu vakuutusmaksu, sekä
   segmentin kantama nykyarvoon diskontattu vastuuvelka, alueittain. */
PROSEDUURI SQL;
    OTSIKKO 'Toteutunut vahinkosuhde ja vastuuvelan nykyarvo alueittain';
    VALITSE region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     MUOTO=dollar12.2,
           sum(premium)                          AS total_premium  MUOTO=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     MUOTO=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve MUOTO=dollar12.2
    FROM rated
    GROUP MUKAAN region
    ORDER MUKAAN region;
QUIT;
OTSIKKO;

                                      Portfolion yhteenveto hinnoittelualueittain                                       

                                                  The MEANS Procedure

                                  Analysis Variable : premium Veloitettu vakuutusmaksu

        Alue                  Mean            Sum
        -----------------------------------------
        esikaupunki         813.04       34147.74
        kaupunki           1987.17       67563.93
        maaseutu            476.61       11438.62
        -----------------------------------------

                                      Analysis Variable : z Uskottavuuskerroin (z)

        Alue                  Mean            Sum
        -----------------------------------------
        esikaupunki           0.68          28.36
        kaupunki              0.70          23.90
        maaseutu              0.71          17.14
        -----------------------------------------

                                 


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Tulosten tulkinta

Hinnoitteleva DATA-askel ei koskaan kirjoita yksittäistä diskontto- tai uskottavuuskaavaa suoraan auki — se vain kutsuu funktioita `pure_premium`, `blend_premium`, `charged_premium` ja `pv_reserve`. Siinä **PROC FCMP:n** hyöty: aktuaariset oletukset elävät yhdessä käännetyssä kirjastossa, jota voidaan yksikkötestata, versionhallita ja käyttää uudelleen hinnoittelu-, vastuuvelka- ja raportointitöissä. Uskottavuusvakion `k`, riskikuormituksen tai kuluasteen muuttaminen on yksi muokkaus kirjastossa, ei etsintää jokaisen ohjelman läpi.

Hinnoitellun kirjan ja alueellisen koonnin lukeminen:

- **Uskottavuus (`z`)** nousee `years_insured`-muuttujan mukana, aivan kuten rajoitetun vaihtelun kaava `Z = sqrt(n / (n + k))` määrää. Ensimmäisten kymmenen vakuutuksen joukossa yhden vuoden esikaupunkivakuutus (10006) saa arvon `z = 0.447`, kahden vuoden kaupunkivakuutus (10003) arvon `z = 0.577`, kun taas yhdeksän vuoden esikaupunkivakuutus (10004) saavuttaa arvon `z = 0.832`. Vähäisen kokemuksen vakuutukset vedetään kohti alueellista luokkamaksua; pitkäaikaiset nojaavat omiin vahinkoihinsa.
- **Painotus käytännössä:** vahingottomilla vakuutuksilla (suurin osa kirjasta) on `own_pp = 0`, joten `blend_premium` palauttaa `blended_pp`-arvon, joka on lähellä `(1 - z)` kertaa luokkamaksu — esim. vakuutus 10001 (maaseutu, `z = 0.745`) asettuu arvoon `91.67` `360`/autovuosi-luokkamaksua vasten. Kaksi kaupunkivakuutusta, joille syntyi vahinkoja, 10002 ja 10003, sen sijaan vetävät painotetun vahinkokustannuksensa ylös kohti omaa korkeaa kokemustaan (`3039.59` ja `3046.88`).
- **Veloitettu vakuutusmaksu** asettuu painotetun vahinkokustannuksen yläpuolelle, koska `charged_premium` lisää 12 %:n riskikuormituksen ja korottaa 25 %:n kuluasteella, kiinteä `1.12 / 0.75 = 1.493` -kerroin vahinkokustannukselle. Keskimääräinen veloitettu vakuutusmaksu on `476.61` (maaseutu), `813.04` (esikaupunki) ja `1,987.17` (kaupunki).
- **Diskontatut vastuuvelat:** `pv_reserve` diskonttaa jokaisen vakuutuksen avoimen korvausvastuun (35 % toteutuneesta vahingosta) kolme vuotta 4 %:n korolla, `0.889`-nykyarvokerroin. Vahingottomilla vakuutuksilla `reserve_pv = 0`; kaksi kaupunkivahinkoa tuovat `1,067.95` ja `2,226.55`. Koottuna kirja sisältää nykyarvoon diskontatun vastuuvelan `$2,151.56` (maaseutu), `$5,932.52` (esikaupunki) ja `$13,364.13` (kaupunki).
- **Toteutuneet vahinkosuhteet** asettuvat tasolle 60.5 % (maaseutu), 55.8 % (esikaupunki) ja 63.6 % (kaupunki) — kaikki mukavasti alle 100 %, joten kuormitettu vakuutusmaksu kattaa odotetun vahingon joka segmentissä. Kaupunkisegmentti on kuumin, kahden suuren simuloidun vahingon ajamana; todellinen hinnoittelutarkastus testaisi, säilyykö tämä signaali useamman vahinkovuoden yli ennen manuaalisen maksun muuttamista.

`blend_premium`-aliohjelma havainnollistaa myös `OUTARGS`-idiomia useamman tuloksen palauttamiseksi yhdestä `CALL`-kutsusta — painotettu vahinkokustannus ja uskottavuuskerroin `z` palautuvat yhdessä — välttäen erilliset funktiokutsut ja pitäen vakuutuskohtaisen hinnoittelulogiikan tiiviinä ja tarkastettavana.